In [1]:
import pandas as pd
import numpy as np 
import tensorflow as tf
import tensorflow_datasets as tfds
print('hello')


hello


In [2]:
#DATAA

mnist_dataset,mnist_info=tfds.load(name='mnist',with_info=True, as_supervised=True)

#spliting the train and test chunks
mnist_train,mnist_test=mnist_dataset['train'], mnist_dataset['test']

#taking 10% of training dataset as validation set
num_validation_samples=0.1* mnist_info.splits['train'].num_examples

#this will overwrite the data into strict integer like converting 5.0 into 5.
num_validation_samples=tf.cast(num_validation_samples,tf.int64)

num_test_samples= mnist_info.splits['test'].num_examples
num_test_samples=tf.cast(num_test_samples,tf.int64)





In [3]:
#It is better to scale our data in some way to make the result more numerically stable.(ex:- inputs between 0 and 1)
def scale(image,label):
    image=tf.cast(image,tf.float32)
    image /=255. #this means we want the img data to be a float 
    return image, label

scaled_train_and_validation_data=mnist_train.map(scale)

scaled_test_data=mnist_test.map(scale)

In [4]:
#shuffle and batch
BUFFER_SIZE= 10000

shuffled_train_and_validation_data=scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

#spliting the scaled and shuffled set for validation and training
validation_data=shuffled_train_and_validation_data.take(num_validation_samples)

test_data=mnist_test.map(scale)

train_data=shuffled_train_and_validation_data.skip(num_validation_samples)

BATCH_SIZE= 100

train_data=train_data.batch(BATCH_SIZE)
validation_data=validation_data.batch(num_validation_samples)

test_data=test_data.batch(num_test_samples)

# takes next batch (it is the only batch)
# because as_supervized=True, we've got a 2-tuple structure

validation_inputs, validation_targets=next(iter(validation_data))

In [16]:
#outlining the model
input_size=784
output_size=10
hidden_layer_size= 100

model=tf.keras.Sequential([
                        tf.keras.Input(shape=(28,28,1)),
                        tf.keras.layers.Flatten(),
                        tf.keras.layers.Dense(hidden_layer_size,activation='relu'),
                        tf.keras.layers.Dense(hidden_layer_size,activation='tanh'),
                        tf.keras.layers.Dense(output_size,activation='softmax')
])


In [ ]:
#selection of loss and optimizer
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [19]:
#model training 
num_epochs=10
model.fit(train_data,epochs=num_epochs,validation_data=(validation_inputs,validation_targets),verbose=2)

Epoch 1/10
540/540 - 6s - 11ms/step - accuracy: 0.9870 - loss: 0.0422 - val_accuracy: 0.9847 - val_loss: 0.0509
Epoch 2/10
540/540 - 6s - 11ms/step - accuracy: 0.9894 - loss: 0.0338 - val_accuracy: 0.9863 - val_loss: 0.0463
Epoch 3/10
540/540 - 6s - 10ms/step - accuracy: 0.9912 - loss: 0.0281 - val_accuracy: 0.9865 - val_loss: 0.0436
Epoch 4/10
540/540 - 6s - 10ms/step - accuracy: 0.9931 - loss: 0.0226 - val_accuracy: 0.9902 - val_loss: 0.0376
Epoch 5/10
540/540 - 6s - 11ms/step - accuracy: 0.9937 - loss: 0.0213 - val_accuracy: 0.9903 - val_loss: 0.0333
Epoch 6/10
540/540 - 6s - 10ms/step - accuracy: 0.9952 - loss: 0.0168 - val_accuracy: 0.9925 - val_loss: 0.0248
Epoch 7/10
540/540 - 6s - 10ms/step - accuracy: 0.9961 - loss: 0.0130 - val_accuracy: 0.9950 - val_loss: 0.0190
Epoch 8/10
540/540 - 6s - 10ms/step - accuracy: 0.9957 - loss: 0.0144 - val_accuracy: 0.9942 - val_loss: 0.0180
Epoch 9/10
540/540 - 6s - 10ms/step - accuracy: 0.9970 - loss: 0.0100 - val_accuracy: 0.9973 - val_loss:

In [20]:
#Testing the model
test_loss,test_accuracy=model.evaluate(test_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9801 - loss: 0.0808


In [24]:
test_inputs,test_targets=next(iter(test_data))
predictions=model.predict(test_inputs)
predicted_digits=np.argmax(predictions,axis=1)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [25]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(test_targets,predicted_digits)
print(cm)

[[ 969    1    1    0    0    2    3    1    2    1]
 [   0 1124    4    1    0    1    2    0    3    0]
 [   6    0 1011    0    1    0    2    4    7    1]
 [   1    0    5  980    0    8    0    5    9    2]
 [   3    0    2    1  960    1    3    1    0   11]
 [   2    0    0    3    1  872    4    0    7    3]
 [   2    2    1    0    6    4  939    0    4    0]
 [   1    0    8    1    0    1    1 1003    5    8]
 [   5    0    0    1    2    2    4    4  953    3]
 [   1    2    0    0    7    2    1    3    3  990]]
